# 🚀 Pre-encode UAReviews with Qwen/Qwen3-Embedding-0.6B

**Project**: Low-Resource Language (Ukrainian) Sentiment & Emotion Analysis  
**Dataset**: [KSE-RESEARCH-Group/UAReviews](https://huggingface.co/datasets/KSE-RESEARCH-Group/UAReviews) (11,580 Ukrainian customer reviews)  
**Embedding Model**: [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) (1024-dim dense representation)  
**Runtime**: PyTorch CUDA GPU (Colab T4 / V100 / A100 or local GPU)

### Overview
This notebook downloads the official Ukrainian Emotion & Intent benchmark `UAReviews`, extracts and verifies the native splits (`train`: 8,106, `test`: 1,737, `challenge`: 1,737), performs text validation, and encodes all 11,580 texts using the state-of-the-art multilingual encoder `Qwen3-Embedding-0.6B`. Embeddings and metadata are cached to Google Drive (with local fallback) in a chunked, resumable format for downstream analysis in `phase_1_3.ipynb` and `phase_4.ipynb`.


In [ ]:
# 1. Install dependencies
!pip install -q "transformers>=4.51.0" "sentence-transformers>=2.7.0" "datasets>=2.18.0" accelerate torch pandas numpy tqdm pyarrow


In [ ]:
# 2. Imports and GPU Verification
import os
import gc
import json
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Hardware verification
print(f"PyTorch version: {torch.__version__}")
CUDA_AVAILABLE = torch.cuda.is_available()
if CUDA_AVAILABLE:
    DEVICE = torch.device("cuda")
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {GPU_NAME} ({VRAM_GB:.2f} GB VRAM)")
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected. Running on CPU (slower).")


In [ ]:
# 3. Environment & Storage Setup (Google Drive with local fallback)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/lrl_sentiment_short  '
    print(f"Mounted Google Drive at: {BASE_DIR}")
except Exception as e:
    BASE_DIR = os.path.abspath('./lrl_cache')
    print(f"Colab drive unavailable ({e}). Using local path: {BASE_DIR}")

CACHE_DIR = os.path.join(BASE_DIR, 'cache')
EMBED_DIR = os.path.join(CACHE_DIR, 'embeddings')
os.makedirs(EMBED_DIR, exist_ok=True)
print(f"Artifacts will be cached in: {CACHE_DIR}")


## 4. Load & Inspect UAReviews Dataset

> **Note on Splits in Hugging Face**:  
> In Hugging Face datasets, `KSE-RESEARCH-Group/UAReviews` stores all 11,580 rows under the default parquet `train` split. The genuine benchmark split partition is defined by the internal `'split'` column, which designates:
> - `'train'`: 8,106 records (70.0%) — training & oversampling pool
> - `'test'`: 1,737 records (15.0%) — in-distribution evaluation holdout
> - `'challenge'`: 1,737 records (15.0%) — out-of-distribution / stress-test holdout


In [ ]:
# 4. Load dataset from Hugging Face
print("Downloading / loading KSE-RESEARCH-Group/UAReviews from Hugging Face...")
raw_ds = load_dataset("KSE-RESEARCH-Group/UAReviews")
df = raw_ds['train'].to_pandas()

print(f"Total records loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Verify split column distribution
split_counts = df['split'].value_counts()
print("\n--- Benchmark Split Counts ---")
for s_name, count in split_counts.items():
    print(f"  {s_name:10s}: {count:5d} ({count / len(df) * 100:.1f}%)")

# Verify emotion distribution
emotion_counts = df['final_emotion'].value_counts()
print("\n--- 7-class Emotion Distribution ---")
for emo, count in emotion_counts.items():
    print(f"  {emo:12s}: {count:5d} ({count / len(df) * 100:.2f}%)")


In [ ]:
# 5. Clean & Validate Data
df['content'] = df['content'].fillna('').astype(str).str.strip()
df['length'] = df['content'].str.len()

initial_len = len(df)
df = df[df['length'] > 0].reset_index(drop=True)
if len(df) < initial_len:
    print(f"Removed {initial_len - len(df)} empty rows.")

EMOTION_LABELS = ['Happiness', 'Anger', 'Neutral', 'Sadness', 'Disgust', 'Surprise', 'Fear']
emotion2id = {name: idx for idx, name in enumerate(EMOTION_LABELS)}
id2emotion = {idx: name for idx, name in enumerate(EMOTION_LABELS)}

df['label'] = df['final_emotion'].map(emotion2id)

clean_df_path = os.path.join(CACHE_DIR, 'ua_reviews_clean.parquet')
df.to_parquet(clean_df_path, index=False)

label_map_path = os.path.join(CACHE_DIR, 'label_encoder.json')
with open(label_map_path, 'w', encoding='utf-8') as f:
    json.dump({'emotion2id': emotion2id, 'id2emotion': id2emotion}, f, indent=2, ensure_ascii=False)

split_indices = {
    'train': df[df['split'] == 'train'].index.tolist(),
    'test': df[df['split'] == 'test'].index.tolist(),
    'challenge': df[df['split'] == 'challenge'].index.tolist()
}
split_indices_path = os.path.join(CACHE_DIR, 'split_indices.json')
with open(split_indices_path, 'w', encoding='utf-8') as f:
    json.dump(split_indices, f, indent=2)

print(f"Cleaned dataset saved: {clean_df_path}")
print(f"Split indices saved: {split_indices_path}")
print(f"Label mapping saved: {label_map_path}")


## 6. Pre-encode with Qwen/Qwen3-Embedding-0.6B

We load `Qwen/Qwen3-Embedding-0.6B` using `SentenceTransformer` with half-precision (`float16`) on CUDA.  
Encoding is performed in chunked batches with automatic checkpointing: if interrupted, running the cell resumes immediately from the cached state.


In [ ]:
# 6. Load Qwen3-Embedding-0.6B Model
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
print(f"Loading embedding model: {MODEL_NAME}...")

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    model_kwargs={"torch_dtype": torch.float16 if CUDA_AVAILABLE else torch.float32}
)

EMB_DIM = model.get_sentence_embedding_dimension()
print(f"Model loaded successfully on {DEVICE}. Embedding dimension: {EMB_DIM}")


In [ ]:
# 7. Resumable Batch Encoding
ALL_EMB_PATH = os.path.join(EMBED_DIR, 'qwen3_embeddings_all.npy')
CHUNK_SIZE = 1000
BATCH_SIZE = 64 if CUDA_AVAILABLE else 16

texts = df['content'].tolist()
n_samples = len(texts)

if os.path.exists(ALL_EMB_PATH):
    print(f"Found existing embeddings at: {ALL_EMB_PATH}")
    all_embeddings = np.load(ALL_EMB_PATH)
    print(f"Loaded embeddings shape: {all_embeddings.shape}")
else:
    print(f"Starting encoding for {n_samples} texts (Batch size: {BATCH_SIZE})...")
    chunks = []
    t_start = time.time()
    
    for chunk_start in range(0, n_samples, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, n_samples)
        chunk_file = os.path.join(EMBED_DIR, f'qwen3_chunk_{chunk_start}_{chunk_end}.npy')
        
        if os.path.exists(chunk_file):
            print(f"  [cached] Chunk {chunk_start}:{chunk_end}")
            chunk_emb = np.load(chunk_file)
        else:
            chunk_texts = texts[chunk_start:chunk_end]
            chunk_emb = model.encode(
                chunk_texts,
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True
            ).astype(np.float32)
            np.save(chunk_file, chunk_emb)
            elapsed = time.time() - t_start
            rate = chunk_end / max(elapsed, 1e-4)
            print(f"  [encoded] Chunk {chunk_start}:{chunk_end} ({rate:.1f} texts/sec)")
        
        chunks.append(chunk_emb)
    
    all_embeddings = np.vstack(chunks).astype(np.float32)
    np.save(ALL_EMB_PATH, all_embeddings)
    print(f"\nAll embeddings successfully encoded and saved: {ALL_EMB_PATH}")
    print(f"Total time: {time.time() - t_start:.1f}s | Shape: {all_embeddings.shape}")


In [ ]:
# 8. Export Split-specific Embeddings & Metadata
train_idx = np.array(split_indices['train'])
test_idx = np.array(split_indices['test'])
chal_idx = np.array(split_indices['challenge'])

X_train = all_embeddings[train_idx]
X_test = all_embeddings[test_idx]
X_chal = all_embeddings[chal_idx]

np.save(os.path.join(EMBED_DIR, 'qwen3_embeddings_train.npy'), X_train)
np.save(os.path.join(EMBED_DIR, 'qwen3_embeddings_test.npy'), X_test)
np.save(os.path.join(EMBED_DIR, 'qwen3_embeddings_challenge.npy'), X_chal)

meta = {
    'model_name': MODEL_NAME,
    'embedding_dim': int(EMB_DIM),
    'n_total': int(n_samples),
    'n_train': int(len(train_idx)),
    'n_test': int(len(test_idx)),
    'n_challenge': int(len(chal_idx)),
    'normalized': True,
    'device': str(DEVICE)
}
with open(os.path.join(CACHE_DIR, 'encoding_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print("Split embeddings saved:")
print(f"  X_train: {X_train.shape} (dtype: {X_train.dtype})")
print(f"  X_test:  {X_test.shape}")
print(f"  X_chal:  {X_chal.shape}")
print(f"Metadata: {meta}")


In [ ]:
# 9. Sanity Check: Cosine Similarity between Sample Ukrainian Sentences
sample_texts = [
    "Дякую, чудовий сервіс і дуже швидке обслуговування!",
    "Дуже вдячний за допомогу працівникам ЦНАП!",
    "Жахливе ставлення до людей, черга на 3 години!",
    "Нічого не працює, повний безлад та хамство!",
]

sample_embs = model.encode(sample_texts, normalize_embeddings=True)
sim_matrix = sample_embs @ sample_embs.T

sim_df = pd.DataFrame(sim_matrix, index=[f"S{i+1}" for i in range(4)], columns=[f"S{i+1}" for i in range(4)])
print("--- Cosine Similarity Sanity Check ---")
print("S1 (Happiness): 'Дякую, чудовий сервіс...'")
print("S2 (Happiness): 'Дуже вдячний за допомогу...'")
print("S3 (Anger):     'Жахливе ставлення до людей...'")
print("S4 (Anger):     'Нічого не працює, повний безлад...'")
print("\nSimilarity Matrix:")
print(sim_df.round(4))
print("\nNotice high within-class similarity (S1-S2, S3-S4) and lower across-class similarity!")
